[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/A.%20Foundations%20of%20Terminal%20Operations%20%28The%20Building%20Blocks%29/06.%20The%20Yard%20Location%20Naming%20Convention%20Problem/P6-Tier-2_executed.ipynb)

---



# 6. The Yard Location Naming Convention Problem

## Tier 2 — The Classic Heuristic (Python Implementation)

### Goal

This notebook implements a hierarchical, constructive naming heuristic for yard locations. Given the terminal's blocks, rows, positions, and maximum tier count, it assigns each physical location a structured identifier of the form `Block-Row-Position-Tier`. It then checks identifier uniqueness and calculates descriptive metrics for length, edit distance, and a defined readability score.

The method proceeds in the hierarchy's order: assign block codes, enumerate the locations within each block, and check each generated identifier against those already used. It is a deterministic naming rule with a conflict-resolution mechanism; it does not search over all possible naming conventions or prove that its identifiers optimize an external objective.

### Inputs, outputs, and scope

- **Inputs:** ordered block names, row counts by block, position counts by block and row, and a maximum tier count.
- **Outputs:** a mapping from each `(block, row, position, tier)` tuple to an identifier, any conflicts that required resolution, and descriptive efficiency metrics.
- **Scope:** the example assumes that rows and positions are numbered consecutively from 1 and that every position has the same maximum number of tiers. It does not model traffic, operator preferences, physical distances, or operational error rates. The readability score is a transparent, manually defined proxy; it is not a measured human-performance outcome.

### Modeling assumptions

The hierarchy is represented by the ordered tuple `(block, row, position, tier)`. The block code is determined by the block's position in the input list: up to 26 blocks receive `A`, `B`, and so on; larger lists receive `B01`, `B02`, and so on. Row and position numbers are zero-padded to two digits, while the tier number is written without padding. If an identifier is already in use, the heuristic appends or advances an alphabetic suffix on its final component until the identifier is unused.

This rule prioritizes a consistent hierarchical structure. No traffic data or other priority measure is supplied, so the implementation does not assign high-traffic locations preferentially.

### Approach

1. Assign a block code to each block in the supplied order.
2. Enumerate each valid `(block, row, position, tier)` combination and construct its identifier.
3. Check each identifier against the set of identifiers already assigned; resolve a collision if one occurs.
4. Calculate identifier-length, edit-distance, uniqueness, and readability metrics.
5. Apply the procedure to the Pacific Gateway Terminal configuration and examine the resulting identifiers and descriptive analyses.

The terminal configuration below contains 5 North rows with 20 positions per row, 8 Central rows with 24 positions per row, and 6 South rows with 18 positions per row. With 4 tiers per position, these inputs define 1,600 locations.

In [ ]:
from collections import defaultdict
from itertools import combinations
import random

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# A consistent plotting style makes the different metric panels easier to compare.
plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

## Hierarchical Greedy Naming Algorithm

### Intuition and role in the solution

A location is identified by progressively specifying where it lies: first its block, then its row, position, and tier. This hierarchy makes the identifier interpretable as a sequence of location components. The heuristic makes each identifier as it visits a location; it does not revise earlier assignments to optimize a global score. The used-identifier set is necessary because two different locations must not receive the same identifier.

### Formal representation

Let the location set be

$$
\mathcal{L} = \{(b,r,p,t): b \in B,\ 1 \leq r \leq R_b,\ 1 \leq p \leq P_{b,r},\ 1 \leq t \leq T\}.
$$

Here, $B$ is the ordered set of blocks, $R_b$ is the row count for block $b$, $P_{b,r}$ is the position count in row $r$ of block $b$, and $T$ is the maximum number of tiers. The implementation represents these values with `blocks`, `rows_per_block`, `positions_per_row`, and `max_tiers`.

For each block, the algorithm assigns a code $c(b)$ according to its position in `blocks`. It constructs the base identifier

$$
I_0(b,r,p,t) = c(b)\texttt{-}r_{2}\texttt{-}p_{2}\texttt{-}t,
$$

where $r_2$ and $p_2$ are two-digit, zero-padded representations of the row and position. In code, this is the formatted string `f"{block_ids[block]}-{row:02d}-{pos:02d}-{tier}"`. If this string is already in the set of assigned identifiers, `resolve_conflict` modifies its final component and the membership check is repeated.

The base rule is to assign the constructed identifier when it is unused. The conflict rule is to keep applying the suffix transformation until an unused identifier is found. The mapping is then extended from the location tuple to that identifier. This is a constructive procedure rather than a recursive dynamic program: it has no Bellman equation or optimization objective.

If $n=|\mathcal{L}|$, generating and storing the $n$ identifiers takes expected $O(n)$ time with average-case set membership, excluding the cost of any repeated conflict resolutions. In a collision-free instance, each location requires one membership check. The edit-distance metric later examines all pairs for small instances, which takes $O(n^2)$ pair comparisons; for larger instances it estimates pairwise distances from a sample of at most 1,000 pairs.

In [ ]:
def levenshtein_distance(s1, s2):
    """Return the Levenshtein edit distance between two strings.

    The distance is the minimum number of single-character insertions,
    deletions, or substitutions needed to transform one string into the other.
    The dynamic program stores one row at a time to limit memory use.
    """
    if len(s1) > len(s2):
        s1, s2 = s2, s1

    distances = list(range(len(s1) + 1))
    for i2, c2 in enumerate(s2):
        new_distances = [i2 + 1]
        for i1, c1 in enumerate(s1):
            if c1 == c2:
                new_distances.append(distances[i1])
            else:
                new_distances.append(
                    1 + min(distances[i1], distances[i1 + 1], new_distances[-1])
                )
        distances = new_distances
    return distances[-1]


def resolve_conflict(identifier):
    """Append or advance an alphabetic suffix on the final component.

    The caller checks whether the returned identifier is unused and repeats
    this transformation if another collision remains.
    """
    parts = identifier.split("-")
    last_part = parts[-1]

    if last_part.isdigit():
        parts[-1] = last_part + "A"
    elif last_part[-1].isalpha():
        base = last_part[:-1]
        suffix = chr(ord(last_part[-1]) + 1)
        parts[-1] = base + suffix
    else:
        parts[-1] = last_part + "A"

    return "-".join(parts)


def get_format_pattern(identifier):
    """Represent letters as A and digits as D, preserving punctuation."""
    return "".join(
        "A" if character.isalpha() else "D" if character.isdigit() else character
        for character in identifier
    )


print("Helper functions are ready.")

## Core Greedy Algorithm Implementation

The function below follows the hierarchy directly. It records assigned identifiers in a set so that a collision check does not require scanning the full mapping each time. Its return values keep the location-to-identifier mapping separate from the record of any resolved conflicts.

In [ ]:
def generate_naming_convention(blocks, rows_per_block, positions_per_row, max_tiers):
    """Generate hierarchical identifiers and record resolved conflicts.

    Args:
        blocks: Ordered list of block identifiers.
        rows_per_block: Mapping from each block to its row count.
        positions_per_row: Mapping from (block, row) to its position count.
        max_tiers: Maximum number of tiers at each position.

    Returns:
        A pair containing the location-to-identifier mapping and a list of
        conflict records. Each location key is (block, row, position, tier).
    """
    location_mapping = {}
    used_ids = set()
    conflicts = []

    # The input order defines block codes, making assignments reproducible.
    block_ids = {}
    for i, block in enumerate(blocks):
        if len(blocks) <= 26:
            block_ids[block] = chr(ord("A") + i)
        else:
            block_ids[block] = f"B{i + 1:02d}"

    for block in blocks:
        for row in range(1, rows_per_block[block] + 1):
            for pos in range(1, positions_per_row[(block, row)] + 1):
                for tier in range(1, max_tiers + 1):
                    identifier = f"{block_ids[block]}-{row:02d}-{pos:02d}-{tier}"
                    original_identifier = identifier
                    conflict_count = 0

                    # Rechecking the set after each suffix change guarantees
                    # that the final identifier is unused.
                    while identifier in used_ids:
                        identifier = resolve_conflict(identifier)
                        conflict_count += 1

                    if conflict_count > 0:
                        conflicts.append(
                            {
                                "location": (block, row, pos, tier),
                                "original": original_identifier,
                                "resolved": identifier,
                                "attempts": conflict_count,
                            }
                        )

                    location_mapping[(block, row, pos, tier)] = identifier
                    used_ids.add(identifier)

    return location_mapping, conflicts


print("The hierarchical naming function is ready.")

## Efficiency Metrics Calculation

Identifier length describes how many characters an operator must read or enter. Levenshtein distance describes the character-level separation between two identifiers: a distance of 1 means that one insertion, deletion, or substitution can turn one string into the other. The minimum and maximum distances summarize the analyzed pairs; they do not, by themselves, measure real-world error probability.

For at most 100 identifiers, the function evaluates every pair. For larger sets, it samples up to 1,000 pair indices to avoid constructing the full set of pairs in memory. Consequently, the reported minimum and maximum distances for a large instance are sample-based estimates, not guaranteed extrema over all pairs.

The readability score is a weighted heuristic. It combines format-pattern consistency, balance between alphabetic and numeric characters, and consistency of identifier lengths. Each component is on a 0–100 scale, and the final score is their weighted average. This score is useful for comparing identifiers under this stated rule, but it is not a validated measure of human readability.

In [ ]:
def calculate_readability_score(identifiers):
    """Return a defined 0–100 readability proxy and its components."""
    if not identifiers:
        raise ValueError("At least one identifier is required to score readability.")

    formats = {get_format_pattern(identifier) for identifier in identifiers}
    consistency = 100.0 if len(formats) == 1 else 100.0 / len(formats)

    total_chars = sum(len(identifier) for identifier in identifiers)
    if total_chars == 0:
        raise ValueError("Identifiers must contain at least one character.")

    alpha_count = sum(
        character.isalpha()
        for identifier in identifiers
        for character in identifier
    )
    digit_count = sum(
        character.isdigit()
        for identifier in identifiers
        for character in identifier
    )
    alpha_ratio = alpha_count / total_chars
    digit_ratio = digit_count / total_chars
    balance = max(0.0, 100.0 - abs(alpha_ratio - digit_ratio) * 200.0)

    lengths = [len(identifier) for identifier in identifiers]
    length_std = float(np.std(lengths))
    length_consistency = max(0.0, 100.0 - length_std * 10.0)

    components = {
        "pattern_consistency": consistency,
        "character_balance": balance,
        "length_consistency": length_consistency,
    }
    score = (
        0.4 * consistency
        + 0.3 * balance
        + 0.3 * length_consistency
    )
    return max(0.0, min(100.0, score)), components


def _pair_from_rank(rank, identifier_count):
    """Map a zero-based rank to a pair in combinations(range(n), 2)."""
    low = 0
    high = identifier_count - 1

    while low < high:
        middle = (low + high + 1) // 2
        pairs_before_middle = middle * (2 * identifier_count - middle - 1) // 2
        if pairs_before_middle <= rank:
            low = middle
        else:
            high = middle - 1

    i = low
    pairs_before_i = i * (2 * identifier_count - i - 1) // 2
    j = i + 1 + rank - pairs_before_i
    return i, j


def calculate_naming_efficiency(location_mapping):
    """Calculate length, pairwise-distance, uniqueness, and readability metrics.

    For more than 100 identifiers, at most 1,000 distinct pair indices are
    sampled. The Python random module controls this sample; no fixed seed is
    specified, so sample-based distance estimates may vary between runs.
    """
    identifiers = list(location_mapping.values())
    n = len(identifiers)
    if n == 0:
        raise ValueError("The location mapping must contain at least one identifier.")

    lengths = [len(identifier) for identifier in identifiers]
    avg_len = sum(lengths) / n
    pair_count = n * (n - 1) // 2

    if pair_count == 0:
        min_dist = None
        max_dist = None
        total_pairs = 0
    else:
        if n <= 100:
            pairs = combinations(identifiers, 2)
        else:
            sample_size = min(1000, pair_count)
            sampled_ranks = random.sample(range(pair_count), sample_size)
            pairs = (
                (identifiers[i], identifiers[j])
                for i, j in (
                    _pair_from_rank(rank, n) for rank in sampled_ranks
                )
            )

        min_dist = float("inf")
        max_dist = 0
        total_pairs = 0
        for id1, id2 in pairs:
            distance = levenshtein_distance(id1, id2)
            min_dist = min(min_dist, distance)
            max_dist = max(max_dist, distance)
            total_pairs += 1

    readability, readability_components = calculate_readability_score(identifiers)
    unique_ids = set(identifiers)

    return {
        "avg_length": avg_len,
        "min_distance": min_dist,
        "max_distance": max_dist,
        "readability": readability,
        "readability_components": readability_components,
        "total_locations": n,
        "unique_identifiers": len(unique_ids),
        "uniqueness_rate": len(unique_ids) / n,
        "pairs_analyzed": total_pairs,
    }


print("Efficiency metric functions are ready.")

## Pacific Gateway Terminal Example

The following configuration supplies the block, row, and position counts used by the naming function. The row counts and the positions per row are defined separately: the former controls which row numbers are generated, and the latter controls how many positions each of those rows contains.

In [ ]:
# Define the terminal configuration.
blocks = ["North", "Central", "South"]
rows_per_block = {"North": 5, "Central": 8, "South": 6}
positions_per_row = {}

positions_by_block = {"North": 20, "Central": 24, "South": 18}
for block in blocks:
    for row in range(1, rows_per_block[block] + 1):
        positions_per_row[(block, row)] = positions_by_block[block]

max_tiers = 4

expected_locations = sum(
    positions_per_row[(block, row)] * max_tiers
    for block in blocks
    for row in range(1, rows_per_block[block] + 1)
)

print("Terminal configuration:")
print(f"Blocks: {blocks}")
print(f"Rows per block: {rows_per_block}")
print(f"Maximum tiers per position: {max_tiers}")
print(f"Sample positions per row: {list(positions_per_row.items())[:3]}")
print(f"Locations defined by this configuration: {expected_locations}")

The next cell applies the heuristic and checks that the number of generated mapping entries matches the number implied by the configuration. The uniqueness assertion verifies the central identifier constraint directly.

In [ ]:
mapping, conflicts = generate_naming_convention(
    blocks,
    rows_per_block,
    positions_per_row,
    max_tiers,
)

assert len(mapping) == expected_locations
assert len(set(mapping.values())) == len(mapping)

print(f"Generated location identifiers: {len(mapping)}")
print(f"Conflicts resolved: {len(conflicts)}")
if conflicts:
    print("Examples of resolved conflicts:")
    for conflict in conflicts[:3]:
        print(
            f"  {conflict['location']}: {conflict['original']} "
            f"-> {conflict['resolved']}"
        )
else:
    print("No conflicts were encountered; all generated identifiers are unique.")

In [ ]:
efficiency = calculate_naming_efficiency(mapping)

print("Naming convention metrics:")
print(f"Total locations: {efficiency['total_locations']}")
print(f"Average identifier length: {efficiency['avg_length']:.2f} characters")
if efficiency["min_distance"] is None:
    print("Pairwise edit distance: not defined for a single identifier")
else:
    print(f"Minimum analyzed edit distance: {efficiency['min_distance']}")
    print(f"Maximum analyzed edit distance: {efficiency['max_distance']}")
print(f"Readability proxy: {efficiency['readability']:.1f}%")
print(f"Uniqueness rate: {efficiency['uniqueness_rate'] * 100:.1f}%")
print(f"Identifier pairs analyzed: {efficiency['pairs_analyzed']}")
if efficiency["total_locations"] > 100:
    print("Pairwise distances are estimated from a random sample of at most 1,000 pairs.")

In [ ]:
# Show examples chosen from the configured blocks and valid row/position ranges.
sample_locations = [
    ("North", 1, 1, 1),
    ("North", 1, 1, 4),
    ("Central", 3, 15, 2),
    ("South", 6, 18, 3),
]

print("Sample location identifiers:")
for location in sample_locations:
    print(f"{location} -> {mapping[location]}")

print("\nIdentifier pattern examples:")
for identifier in list(mapping.values())[:10]:
    print(f"  {identifier:<12} -> Pattern: {get_format_pattern(identifier)}")

## Visualization of Results

These panels describe the identifiers generated by the configured instance. The readability-component panel uses the same formulas as the reported score rather than illustrative estimates. The normalized-metrics panel rescales unlike quantities for display, so its bar heights should be compared only within that plot, not interpreted as values in a common physical unit.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Identifier lengths show how many characters the generated codes contain.
lengths = [len(identifier) for identifier in mapping.values()]
axes[0, 0].hist(lengths, bins=20, alpha=0.7, color="skyblue", edgecolor="black")
axes[0, 0].set_title("Identifier Length Distribution")
axes[0, 0].set_xlabel("Identifier length (characters)")
axes[0, 0].set_ylabel("Number of identifiers")
axes[0, 0].axvline(
    x=np.mean(lengths),
    color="red",
    linestyle="--",
    label=f"Mean: {np.mean(lengths):.1f}",
)
axes[0, 0].legend()

# Block counts show how the configured location capacity is distributed.
block_counts = defaultdict(int)
for block, _, _, _ in mapping:
    block_counts[block] += 1
axes[0, 1].bar(list(block_counts), list(block_counts.values()))
axes[0, 1].set_title("Locations per Block")
axes[0, 1].set_xlabel("Block")
axes[0, 1].set_ylabel("Number of locations")

# Tier counts expose the distribution across vertical storage levels.
tier_counts = defaultdict(int)
for _, _, _, tier in mapping:
    tier_counts[tier] += 1
axes[0, 2].bar(list(tier_counts), list(tier_counts.values()))
axes[0, 2].set_title("Locations per Tier")
axes[0, 2].set_xlabel("Tier")
axes[0, 2].set_ylabel("Number of locations")

# These values are the components used by calculate_readability_score.
components = efficiency["readability_components"]
component_labels = [
    "Pattern consistency",
    "Character balance",
    "Length consistency",
]
component_values = [
    components["pattern_consistency"],
    components["character_balance"],
    components["length_consistency"],
]
axes[1, 0].bar(component_labels, component_values)
axes[1, 0].set_title("Readability Proxy Components")
axes[1, 0].set_ylabel("Score (0–100)")
axes[1, 0].tick_params(axis="x", rotation=35)
axes[1, 0].set_ylim(0, 100)

# Normalization makes the magnitudes visible together but removes their units.
metric_labels = [
    "Locations",
    "Average length",
    "Minimum distance",
    "Readability",
]
metric_values = [
    efficiency["total_locations"],
    efficiency["avg_length"],
    efficiency["min_distance"] or 0,
    efficiency["readability"],
]
maximum_metric = max(metric_values)
normalized_values = [value / maximum_metric for value in metric_values]
axes[1, 1].bar(metric_labels, normalized_values)
axes[1, 1].set_title("Normalized Descriptive Metrics")
axes[1, 1].set_xlabel("Metric")
axes[1, 1].set_ylabel("Normalized value")
axes[1, 1].tick_params(axis="x", rotation=35)

# Each line represents the number of locations in each row of one block.
row_by_block = defaultdict(lambda: defaultdict(int))
for block, row, _, _ in mapping:
    row_by_block[block][row] += 1
for block, rows in row_by_block.items():
    row_numbers = sorted(rows)
    axes[1, 2].plot(
        row_numbers,
        [rows[row] for row in row_numbers],
        marker="o",
        label=block,
        linewidth=2,
    )
axes[1, 2].set_title("Locations per Row by Block")
axes[1, 2].set_xlabel("Row number")
axes[1, 2].set_ylabel("Number of locations")
axes[1, 2].legend(title="Block")
axes[1, 2].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## Algorithm Analysis and Comparison

### What this tier provides

This tier provides a direct, code-level construction rule: traverse the spatial hierarchy, generate a structured identifier, and enforce uniqueness with a set membership check and a suffix-based conflict rule. Its output is immediately available as a Python mapping and can be inspected with the metrics defined in this notebook.

### Relationship to a mathematical formulation

A mathematical formulation can express requirements such as identifier uniqueness and formatting as constraints, and can define an objective if a measurable criterion is supplied. This implementation does not solve such an optimization model. It applies a fixed hierarchical rule and reports descriptive measures afterward. Therefore, it offers no guarantee that its identifiers are globally optimal under any objective, including the readability proxy.

### Computational considerations

For $n$ locations, identifier generation is expected to take $O(n)$ time when set operations take average constant time and conflicts are absent. The number of generated identifiers and the mapping storage are both $O(n)$. Conflict resolution adds work when collisions occur. Pairwise edit-distance analysis is $O(n^2)$ in the exact small-instance branch; for larger instances, the notebook evaluates at most 1,000 sampled pairs, with the cost of each comparison depending on the identifier lengths.

The heuristic's simple structure makes its assignment rule easy to inspect. Its suitability for operations still depends on requirements not represented here, such as local naming standards, human recognition, equipment interfaces, and how identifiers are used in practice.

In [ ]:
import time

# These configurations illustrate how the same naming rule behaves at
# increasing sizes; elapsed time is measured when this cell is run.
test_configs = [
    {"blocks": ["A", "B"], "rows": 3, "positions": 4, "tiers": 2, "name": "Small"},
    {"blocks": ["A", "B", "C"], "rows": 5, "positions": 8, "tiers": 3, "name": "Medium"},
    {
        "blocks": ["A", "B", "C", "D"],
        "rows": 8,
        "positions": 15,
        "tiers": 4,
        "name": "Large",
    },
    {
        "blocks": ["A", "B", "C", "D", "E"],
        "rows": 10,
        "positions": 20,
        "tiers": 4,
        "name": "X-Large",
    },
]

print("Performance analysis for different terminal sizes:")
print("=" * 76)
print(
    f"{'Terminal size':<15} {'Locations':>10} {'Avg length':>12} "
    f"{'Readability':>13} {'Time (ms)':>12}"
)
print("=" * 76)

for config in test_configs:
    test_blocks = config["blocks"]
    test_rows_per_block = {
        block: config["rows"] for block in test_blocks
    }
    test_positions_per_row = {
        (block, row): config["positions"]
        for block in test_blocks
        for row in range(1, config["rows"] + 1)
    }

    start_time = time.perf_counter()
    test_mapping, test_conflicts = generate_naming_convention(
        test_blocks,
        test_rows_per_block,
        test_positions_per_row,
        config["tiers"],
    )
    elapsed_ms = (time.perf_counter() - start_time) * 1000
    test_efficiency = calculate_naming_efficiency(test_mapping)

    print(
        f"{config['name']:<15} "
        f"{test_efficiency['total_locations']:>10} "
        f"{test_efficiency['avg_length']:>12.2f} "
        f"{test_efficiency['readability']:>12.1f}% "
        f"{elapsed_ms:>11.2f}"
    )

## Summary and Interpretation

The Pacific Gateway example applies the same hierarchical rule to every location defined by its block, row, position, and tier inputs. The generated mapping and the metric values are calculated by the notebook when its code is run; they are not assumed in advance here.

Use the results as follows:

- **Generated location count** should match the count implied by the configuration. The assertion checks this relationship.
- **Uniqueness rate** measures the share of mapping values that are distinct. A value of 100% means that every generated identifier is unique within this mapping.
- **Average length** summarizes identifier size; it does not account for how easily people distinguish identifiers.
- **Minimum and maximum edit distances** summarize the analyzed pairs. For more than 100 identifiers, they are sample-based and may vary between runs.
- **Readability proxy** reflects only the explicitly defined pattern, character-balance, and length-consistency rules. It should not be interpreted as a measured operational performance score.
- **Timing results** describe the execution environment and configuration used when the experiment cell runs; they are not universal performance guarantees.

The central lesson is that a transparent hierarchical heuristic can generate structured identifiers and enforce uniqueness through a simple construction process. Whether its naming convention is suitable for a particular terminal requires operational requirements and human evaluation beyond the metrics modeled here.